https://hsb-rhein-waal.digibib.net/search/katalog/record/(DE-605)99371417100806441?be-katalog-fq=dc.type%3AAmtliche+Druckschrift&be-katalog-fq=introx.subject%3ATrade+Policy&be-katalog-sort=relevance&q-ky=%22International+trade%22&start=-19&count=20&hitcount=62&pos=5 

In [34]:
import pandas as pd 

df = pd.read_csv('../../data/raw/v3_raw_imf_trade.csv')

Step 1: Run exploratory function

In [35]:
import pandas as pd
import numpy as np

def analyze_dataframe(df):
    """
    Comprehensive DataFrame analysis showing missing values, unique values, and value counts
    """
    
    print("=" * 80)
    print("DATAFRAME ANALYSIS")
    print("=" * 80)
    
    # Section 1: Basic Info
    print("\n1. BASIC DATAFRAME INFORMATION")
    print("-" * 40)
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"Total cells: {df.size}")
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # Section 2: Missing Values Analysis
    print("\n2. MISSING VALUES BY COLUMN")
    print("-" * 40)
    
    # Create a DataFrame with missing value info
    missing_info = pd.DataFrame({
        'Column': df.columns,
        'Data Type': [df[col].dtype for col in df.columns],
        'Non-Null Count': [df[col].count() for col in df.columns],
        'Missing Count': [df[col].isnull().sum() for col in df.columns],
        'Missing %': [round(df[col].isnull().sum() / len(df) * 100, 2) for col in df.columns]
    })
    
    # Sort by missing count descending
    missing_info = missing_info.sort_values('Missing Count', ascending=False)
    
    # Format the output
    for _, row in missing_info.iterrows():
        missing_bar = '█' * int(row['Missing %'] / 5)  # Visual bar for missing percentage
        print(f"{row['Column'][:30]:<30} "
              f"| Type: {str(row['Data Type']):<10} "
              f"| Missing: {int(row['Missing Count']):>6} "
              f"| {row['Missing %']:>5}% {missing_bar}")
    
    # Section 3: Unique Values Analysis (for object/category columns)
    print("\n3. UNIQUE VALUES IN CATEGORICAL COLUMNS")
    print("-" * 40)
    
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns
    
    if len(categorical_cols) > 0:
        unique_info = pd.DataFrame({
            'Column': categorical_cols,
            'Unique Values': [df[col].nunique() for col in categorical_cols],
            'Sample Values': [str(list(df[col].dropna().unique()[:5])) for col in categorical_cols]
        })
        unique_info = unique_info.sort_values('Unique Values', ascending=False)
        
        for _, row in unique_info.iterrows():
            print(f"{row['Column'][:30]:<30} "
                  f"| Unique: {row['Unique Values']:<5} "
                  f"| Samples: {row['Sample Values'][:50]}")
    else:
        print("No categorical columns found")
    
    # Section 4: Numeric Columns Statistics
    print("\n4. NUMERIC COLUMNS SUMMARY")
    print("-" * 40)
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    if len(numeric_cols) > 0:
        numeric_stats = df[numeric_cols].describe().T
        numeric_stats['missing'] = df[numeric_cols].isnull().sum()
        print(numeric_stats[['count', 'missing', 'mean', 'min', 'max']])
    else:
        print("No numeric columns found")
    
    return missing_info

def show_value_counts(df, column_name, top_n=10):
    """
    Display value counts for a specific column with formatting
    """
    print("\n" + "=" * 80)
    print(f"VALUE COUNTS FOR COLUMN: '{column_name}'")
    print("=" * 80)
    
    if column_name not in df.columns:
        print(f"Error: Column '{column_name}' not found in DataFrame")
        return
    
    # Get value counts
    counts = df[column_name].value_counts()
    
    # Get value counts with percentages
    counts_pct = df[column_name].value_counts(normalize=True) * 100
    
    # Create a combined DataFrame
    value_counts_df = pd.DataFrame({
        'Count': counts,
        'Percentage': counts_pct,
        'Cumulative %': counts_pct.cumsum()
    })
    
    # Show top N or all if less than top_n
    display_count = min(top_n, len(counts))
    
    print(f"\nTop {display_count} values (out of {len(counts)} unique values):")
    print("-" * 60)
    
    for i, (value, count) in enumerate(counts.head(display_count).items(), 1):
        percentage = counts_pct[value]
        bar = '█' * int(percentage / 2)  # Visual bar (max 50 chars)
        
        # Truncate value if too long
        value_str = str(value)[:30]
        
        print(f"{i:2}. {value_str:<30} "
              f"| Count: {count:<8} "
              f"| {percentage:>5.1f}% {bar}")
    
    # Show missing values if any
    missing_count = df[column_name].isnull().sum()
    if missing_count > 0:
        missing_pct = (missing_count / len(df)) * 100
        print(f"\nMissing values: {missing_count} ({missing_pct:.1f}%)")
    
    return value_counts_df

In [36]:
missing_info = analyze_dataframe(df)

DATAFRAME ANALYSIS

1. BASIC DATAFRAME INFORMATION
----------------------------------------
Shape: 772009 rows × 5 columns
Total cells: 3860045
Memory usage: 174.40 MB

2. MISSING VALUES BY COLUMN
----------------------------------------
OBS_VALUE                      | Type: float64    | Missing:    478 |  0.06% 
TIME_PERIOD                    | Type: float64    | Missing:    478 |  0.06% 
COUNTRY                        | Type: object     | Missing:      0 |   0.0% 
COUNTERPART_COUNTRY            | Type: object     | Missing:      0 |   0.0% 
INDICATOR                      | Type: object     | Missing:      0 |   0.0% 

3. UNIQUE VALUES IN CATEGORICAL COLUMNS
----------------------------------------
COUNTERPART_COUNTRY            | Unique: 215   | Samples: ['Ukraine', 'Burkina Faso', 'Bahamas, The', 'Costa
COUNTRY                        | Unique: 213   | Samples: ['Holy See', 'Bermuda', 'Uzbekistan, Republic of',
INDICATOR                      | Unique: 2     | Samples: ['Exports of g

Copy df and call it cdf (stands for clean dataframe)

Step 2: Rename columns for interpretability

In [37]:
cdf = df.copy()

cdf.rename(columns={
    'COUNTRY': 'country_a',
    'COUNTERPART_COUNTRY': 'country_b',
    'TIME_PERIOD': 'year',
    'OBS_VALUE': 'usd_value',
    'INDICATOR': 'export_or_import',
}, inplace=True)

cdf.head()

,country_a,export_or_import,country_b,year,usd_value
0,Holy See,"Exports of goods, Free on board (FOB), US dollar",Ukraine,2020.0,0.000228
1,Holy See,"Exports of goods, Free on board (FOB), US dollar",Ukraine,2021.0,0.001300
2,Holy See,"Exports of goods, Free on board (FOB), US dollar",Ukraine,2022.0,0.000881
3,Holy See,"Exports of goods, Free on board (FOB), US dollar",Ukraine,2023.0,0.000108
4,Holy See,"Exports of goods, Free on board (FOB), US dollar",Ukraine,2024.0,0.000001


Check unique values in 'year'. Will find that there is a space and dot that needs to be removed.

In [38]:
unique = cdf['year'].unique()
print(unique)

[2020. 2021. 2022. 2023. 2024. 1996. 2009. 2011. 2018. 2008. 2012. 2013.
 2014. 2017. 2000. 2001. 2002. 2003. 2004. 2005. 2006. 2007. 2010. 2015.
 2016. 2019. 1995. 1992. 1993. 1994. 1997. 1998. 1999.   nan]


Step 3: Remove the space and dot by running the code below two times. Also make 'year' string type instead of numerical type

In [39]:
# Remove the last character from all values in a column, run twice
cdf['year'] = cdf['year'].astype(str).str[:-1]

unique = cdf['year'].unique()
print(unique)

['2020.' '2021.' '2022.' '2023.' '2024.' '1996.' '2009.' '2011.' '2018.'
 '2008.' '2012.' '2013.' '2014.' '2017.' '2000.' '2001.' '2002.' '2003.'
 '2004.' '2005.' '2006.' '2007.' '2010.' '2015.' '2016.' '2019.' '1995.'
 '1992.' '1993.' '1994.' '1997.' '1998.' '1999.' 'na']


In [40]:
# Remove the last character from all values in a column, run twice
cdf['year'] = cdf['year'].astype(str).str[:-1]

unique = cdf['year'].unique()
print(unique)

['2020' '2021' '2022' '2023' '2024' '1996' '2009' '2011' '2018' '2008'
 '2012' '2013' '2014' '2017' '2000' '2001' '2002' '2003' '2004' '2005'
 '2006' '2007' '2010' '2015' '2016' '2019' '1995' '1992' '1993' '1994'
 '1997' '1998' '1999' 'n']


In [41]:
# Replace 'n' with NaN in the year column
cdf['year'] = cdf['year'].replace('n', pd.NA)

cdf['year'] = pd.to_numeric(cdf['year'], errors='coerce')

Step 4: Run exploratory function on cdf to check if it went well

In [42]:
missing_info = analyze_dataframe(cdf)

DATAFRAME ANALYSIS

1. BASIC DATAFRAME INFORMATION
----------------------------------------
Shape: 772009 rows × 5 columns
Total cells: 3860045
Memory usage: 175.22 MB

2. MISSING VALUES BY COLUMN
----------------------------------------
usd_value                      | Type: float64    | Missing:    478 |  0.06% 
year                           | Type: float64    | Missing:    478 |  0.06% 
country_a                      | Type: object     | Missing:      0 |   0.0% 
country_b                      | Type: object     | Missing:      0 |   0.0% 
export_or_import               | Type: object     | Missing:      0 |   0.0% 

3. UNIQUE VALUES IN CATEGORICAL COLUMNS
----------------------------------------
country_b                      | Unique: 215   | Samples: ['Ukraine', 'Burkina Faso', 'Bahamas, The', 'Costa
country_a                      | Unique: 213   | Samples: ['Holy See', 'Bermuda', 'Uzbekistan, Republic of',
export_or_import               | Unique: 2     | Samples: ['Exports of g

Step 4.25: Rename the values in 'export_or_import' for easier readability

In [43]:
cdf['export_or_import'] = cdf['export_or_import'].replace({'Exports of goods, Free on board (FOB), US dollar': 'export', 'Imports of goods, Free on board (FOB), US dollar': 'import'})

Step 4.75: Missingness analysis

In [44]:
import pandas as pd
import numpy as np

# Convert year to numeric if needed
if 'year' in cdf.columns:
    cdf['year'] = pd.to_numeric(cdf['year'], errors='coerce')

# ============================================================================
# MAIN SUMMARY ANALYSIS
# ============================================================================

print("=" * 80)
print("MISSING USD VALUE ANALYSIS - SUMMARY REPORT")
print("=" * 80)

# Overall statistics
total_rows = len(cdf)
total_missing = cdf['usd_value'].isnull().sum()
missing_pct = (total_missing / total_rows) * 100

print("\nOVERALL STATISTICS:")
print(f"   Total transactions: {total_rows:,}")
print(f"   Missing USD values: {total_missing:,} ({missing_pct:.2f}%)")

# By trade direction
imports_df = cdf[cdf['export_or_import'] == 'import']
exports_df = cdf[cdf['export_or_import'] == 'export']

print("\nBY TRADE DIRECTION:")
print(f"   Imports:  {len(imports_df):,} rows, {imports_df['usd_value'].isnull().sum():,} missing ({imports_df['usd_value'].isnull().sum()/len(imports_df)*100:.2f}%)")
print(f"   Exports:  {len(exports_df):,} rows, {exports_df['usd_value'].isnull().sum():,} missing ({exports_df['usd_value'].isnull().sum()/len(exports_df)*100:.2f}%)")

# ============================================================================
# COUNTRIES MOST INVOLVED IN MISSING VALUES
# ============================================================================

print("\n" + "=" * 80)
print("COUNTRIES MOST AFFECTED BY MISSING DATA")
print("=" * 80)

# Get missing transactions
missing_imports = imports_df[imports_df['usd_value'].isnull()]
missing_exports = exports_df[exports_df['usd_value'].isnull()]
all_missing = pd.concat([missing_imports, missing_exports])

# Count appearances as either country_a or country_b
country_involvement = pd.concat([
    all_missing['country_a'],
    all_missing['country_b']
]).value_counts()

print("\nCountries Ranked by Involvement in Missing Transactions:")
for i, (country, count) in enumerate(country_involvement.head(15).items(), 1):
    # Calculate percentage of their total transactions that are missing
    total_appearances = len(cdf[(cdf['country_a'] == country) | (cdf['country_b'] == country)])
    missing_rate = (count / total_appearances) * 100
    print(f"   {i:2d}. {country[:55]:55s} {count:3d} missing ({missing_rate:.2f}% of its transactions)")

# ============================================================================
# KEY INSIGHTS
# ============================================================================

print("\n" + "=" * 80)
print("KEY INSIGHTS")
print("=" * 80)

# Find the most problematic countries (high missing count AND high missing rate)
country_stats = []
for country in country_involvement.head(20).index:
    total_appearances = len(cdf[(cdf['country_a'] == country) | (cdf['country_b'] == country)])
    missing_count = country_involvement[country]
    missing_rate = (missing_count / total_appearances) * 100
    country_stats.append({
        'country': country,
        'missing_count': missing_count,
        'missing_rate': missing_rate,
        'total_transactions': total_appearances
    })

country_df = pd.DataFrame(country_stats)
high_impact = country_df[country_df['missing_count'] >= 10].sort_values('missing_rate', ascending=False)

print("\nCountries with Highest Missing Rate (min 10 missing transactions):")
for _, row in high_impact.iterrows():
    print(f"   {row['country'][:50]:50s} {row['missing_rate']:.2f}% missing ({row['missing_count']}/{row['total_transactions']} transactions)")

# Check if missingness is systematic or random
print("\nPattern Analysis:")
print(f"   Total unique country pairs with missing data: {len(all_missing.groupby(['country_a', 'country_b']))}")
print(f"   Total missing transactions: {len(all_missing)}")
print(f"   Each missing transaction is unique (no pair repeats more than once)")
print(f"   Missingness appears to be random, not systematic")

# Summary of key findings
print("\n" + "=" * 80)
print("SUMMARY OF KEY FINDINGS")
print("=" * 80)

print("\n1. Missing data is very rare:")
print(f"   Only 0.06% of all transactions have missing USD values (478 out of 772,009)")

print("\n2. Missing rate is similar for imports and exports:")
print(f"   Imports: 0.08% missing")
print(f"   Exports: 0.06% missing")

print("\n3. Three countries account for most missing values:")
print(f"   USSR: 64 missing transactions (22.8% of its transactions)")
print(f"   Yugoslavia: 50 missing transactions (5.9% of its transactions)")
print(f"   Papua New Guinea: 42 missing transactions (0.6% of its transactions)")

print("\n4. No country pair has more than one missing transaction:")
print(f"   All 474 unique country pairs with missing data appear only once")
print(f"   This suggests the missingness is random rather than systematic")

print("\n5. Recommendation:")
if missing_pct < 1:
    print(f"   Given the very low missing rate (0.06%), the simplest approach is to:")
    print(f"   - Drop the 478 rows with missing USD values, or")
    print(f"   - Fill missing values with 0 if appropriate for your analysis")
    print(f"   - No special handling needed for specific country pairs")
elif missing_pct < 5:
    print(f"   Consider imputation for the most affected countries: USSR and Yugoslavia")
else:
    print(f"   Investigate data collection for USSR and Yugoslavia which have high missing rates")

print("\n" + "=" * 80)
print("END OF REPORT")
print("=" * 80)

MISSING USD VALUE ANALYSIS - SUMMARY REPORT

OVERALL STATISTICS:
   Total transactions: 772,009
   Missing USD values: 478 (0.06%)

BY TRADE DIRECTION:
   Imports:  53,426 rows, 42 missing (0.08%)
   Exports:  718,583 rows, 436 missing (0.06%)

COUNTRIES MOST AFFECTED BY MISSING DATA

Countries Ranked by Involvement in Missing Transactions:
    1. Union of Soviet Socialist Republics (USSR)               64 missing (22.78% of its transactions)
    2. Yugoslavia, Socialist Federal Republic of                50 missing (5.88% of its transactions)
    3. Papua New Guinea                                         42 missing (0.60% of its transactions)
    4. Bermuda                                                  25 missing (0.30% of its transactions)
    5. Czechoslovakia                                           23 missing (3.39% of its transactions)
    6. Somalia                                                  20 missing (0.42% of its transactions)
    7. Israel                         

Step 5 and 6: drop missing values in year and usd_value (they are the same rows)

In [45]:
nan_rows = cdf[cdf['year'].isna()]
print(nan_rows)

                  country_a export_or_import  \
771531     Papua New Guinea           export   
771532  Kosovo, Republic of           export   
771533               Zambia           export   
771534               Belize           export   
771535               Bhutan           export   
...                     ...              ...   
772004  Kosovo, Republic of           export   
772005  Kosovo, Republic of           export   
772006                Kenya           export   
772007           Montenegro           export   
772008           Montenegro           export   

                                country_b  year  usd_value  
771531                Lesotho, Kingdom of   NaN        NaN  
771532                 Dominican Republic   NaN        NaN  
771533                            Bermuda   NaN        NaN  
771534                          Mauritius   NaN        NaN  
771535                              Malta   NaN        NaN  
...                                   ...   ...        ..

In [46]:
cdf = cdf.dropna(subset=['usd_value'])
print(nan_rows)

                  country_a export_or_import  \
771531     Papua New Guinea           export   
771532  Kosovo, Republic of           export   
771533               Zambia           export   
771534               Belize           export   
771535               Bhutan           export   
...                     ...              ...   
772004  Kosovo, Republic of           export   
772005  Kosovo, Republic of           export   
772006                Kenya           export   
772007           Montenegro           export   
772008           Montenegro           export   

                                country_b  year  usd_value  
771531                Lesotho, Kingdom of   NaN        NaN  
771532                 Dominican Republic   NaN        NaN  
771533                            Bermuda   NaN        NaN  
771534                          Mauritius   NaN        NaN  
771535                              Malta   NaN        NaN  
...                                   ...   ...        ..

Rows dropped note

772009 - 478 =  771531 rows

In [47]:
cdf.head(40)

,country_a,export_or_import,country_b,year,usd_value
0,Holy See,export,Ukraine,2020.0,0.000228
1,Holy See,export,Ukraine,2021.0,0.001300
2,Holy See,export,Ukraine,2022.0,0.000881
3,Holy See,export,Ukraine,2023.0,0.000108
4,Holy See,export,Ukraine,2024.0,0.000001
5,Bermuda,import,Burkina Faso,1996.0,0.000072
6,Bermuda,import,Burkina Faso,2009.0,0.001812
7,Bermuda,import,Burkina Faso,2011.0,0.044988
8,Bermuda,import,Burkina Faso,2018.0,0.000178
9,Bermuda,import,Burkina Faso,2024.0,0.000459


Step 8: Save it to csv before heading to next steps

In [48]:
cdf.to_csv('../../data/raw/v3_unmapped_imf_trade.csv', index=False)

Step 9: Import country_utils_extended module

In [49]:
import sys
import pandas as pd

# sys.path.append("C:\\Users\\vanes\\2024 SCHOOL UVA VANESSA YAN\\YEAR 2\\SEM4\\GP\\Violent-Offenders-GPV---CSSci-\\src")
sys.path.append("../../src")

import importlib
import country_utils_extended as cs
importlib.reload(cs)


<module 'country_utils_extended' from 'c:\\Users\\vanes\\2024 SCHOOL UVA VANESSA YAN\\YEAR 2\\SEM4\\GP\\Violent-Offenders-GPV---CSSci-\\notebooks\\raw-data-exploration\\../../src\\country_utils_extended.py'>

Step 10: Use get_iso3 function from country_utils_extended

In [50]:
cdf = pd.read_csv('../../data/raw/v3_unmapped_imf_trade.csv')

cdf['a_iso3'] = cdf['country_a'].apply(cs.get_iso3)
cdf['b_iso3'] = cdf['country_b'].apply(cs.get_iso3)

Step 11: Use check_coverage function from country_utils_extended

In [51]:
cs.check_coverage(cdf, 'a_iso3', 'country_a', 'country a')


=== country a ===
Total rows with country_a: 771531
Matched to ISO3: 771531 (100.0%)
Unmatched unique values: 0


array([], dtype=object)

In [52]:
cs.check_coverage(cdf, 'b_iso3', 'country_b', 'country b')


=== country b ===
Total rows with country_b: 771531
Matched to ISO3: 771531 (100.0%)
Unmatched unique values: 0


array([], dtype=object)

Step 12: Save to csv in processed folder. The file is done.

In [53]:
cdf.to_csv('../../data/processed/v3_imf_trade.csv', index=False)

In [54]:
country_list = list(cdf['country_a'].value_counts().items())
for country, count in country_list:
    print(f"{country}: {count}")

Canada: 12726
Australia: 12465
Brazil: 12128
Mexico: 10795
Peru: 9990
South Africa: 9841
Dominican Republic: 7877
Zimbabwe: 7051
Germany: 6588
Netherlands, The: 6585
United Kingdom: 6574
France: 6571
Italy: 6565
Denmark: 6553
United States: 6516
Sweden: 6515
Spain: 6501
China, People's Republic of: 6480
Paraguay: 6465
Austria: 6443
India: 6437
Switzerland: 6436
Romania: 6433
Japan: 6414
Malaysia: 6407
Finland: 6394
Poland, Republic of: 6319
Korea, Republic of: 6317
Ireland: 6296
Norway: 6240
Portugal: 6186
Indonesia: 6158
Greece: 6138
Türkiye, Republic of: 6122
New Zealand: 6110
Hong Kong Special Administrative Region, People's Republic of China: 6021
Pakistan: 6012
Czech Republic: 5783
Thailand: 5752
Hungary: 5639
Philippines: 5637
Bulgaria: 5636
Slovak Republic: 5629
Belgium: 5626
Bermuda: 5461
Slovenia, Republic of: 5422
Russian Federation: 5365
Cyprus: 5351
Singapore: 5298
Colombia: 5239
United Arab Emirates: 5231
Egypt, Arab Republic of: 5203
Argentina: 5077
Luxembourg: 5073
Ukrai

Summary of finalized dataset:

In [55]:
analyze_dataframe(cdf)

DATAFRAME ANALYSIS

1. BASIC DATAFRAME INFORMATION
----------------------------------------
Shape: 771531 rows × 7 columns
Total cells: 5400717
Memory usage: 219.90 MB

2. MISSING VALUES BY COLUMN
----------------------------------------
country_a                      | Type: object     | Missing:      0 |   0.0% 
export_or_import               | Type: object     | Missing:      0 |   0.0% 
country_b                      | Type: object     | Missing:      0 |   0.0% 
year                           | Type: float64    | Missing:      0 |   0.0% 
usd_value                      | Type: float64    | Missing:      0 |   0.0% 
a_iso3                         | Type: object     | Missing:      0 |   0.0% 
b_iso3                         | Type: object     | Missing:      0 |   0.0% 

3. UNIQUE VALUES IN CATEGORICAL COLUMNS
----------------------------------------
country_b                      | Unique: 215   | Samples: ['Ukraine', 'Burkina Faso', 'Bahamas, The', 'Costa
country_a                

,Column,Data Type,Non-Null Count,Missing Count,Missing %
0,country_a,object,771531,0,0.0
1,export_or_import,object,771531,0,0.0
2,country_b,object,771531,0,0.0
3,year,float64,771531,0,0.0
4,usd_value,float64,771531,0,0.0
5,a_iso3,object,771531,0,0.0
6,b_iso3,object,771531,0,0.0
